# Model 2 — Predicting the defect at the NEXT inspection of the same vehicle (Dutch RDW histories)  (experiments only)

**Question.** Notebooks 04–06 predicted whether the inspection *at a given age* finds a defect (cross-sectional). The original goal is *future* failures. Here the unit is a **pair of consecutive inspections of the same car**:
everything known up to inspection *i* (the car's registry fields and the results of all its earlier inspections) is used to predict whether inspection *i+1* finds a defect in **brakes**, **electrical** (lighting, battery, wiring) or **engine-related** items (emissions, fuel system, mounts).

**What this is and is not.** The target is still a defect *found at an inspection*, but now it lies **in the future** with respect to the features, and the car's own history is available. It is not a breakdown: nothing between the two inspections (repairs, failures, mileage) is observed.

**Leakage controls.**
* Features come only from inspections **up to and including *i***. The target is the result of inspection *i+1*.
* **Temporal split:** training pairs have a target date before 1 Jan 2026; the test pairs have a target date on or after 1 Jan 2026 (to 16 Nov 2026).
* **Cars are disjoint:** cars are split 70 / 15 / 15 at random; train, validation (threshold choice) and test never share a plate. A second check scores the *training cars'* 2026 pairs to see whether remembering a car inflates the result.
* No re-inspections (< 60 days) are counted as inspections. No future date other than the planned gap is used (see the variants).
* Three different random car splits are run; spread is reported.

**Data limit.** The open RDW file of inspection notifications effectively starts in Q4 2023, so histories are 1 to 3 inspections long (2 years of cars for most). The Dutch data have no odometer; the odometer question is tested with the UK data in the next notebook.

In [1]:
import sys, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
pd.options.display.width = 190; pd.options.display.max_columns = 40; pd.options.display.float_format = "{:,.4f}".format
sys.path.insert(0, str(Path.cwd().parent / "src"))
from real_outcome_data import load_both
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, SplineTransformer
from xgboost import XGBClassifier
try:
    display
except NameError:
    display = print
_, nl = load_both()
SYS = ["brakes", "electrical", "engine_emissions"]
d = nl.sort_values(["unit", "date"]).copy()
g = d.groupby("unit")
d["n_insp"] = g.cumcount() + 1                                     # inspections of this car observed up to and including this one
d["days_since_prev"] = g.date.diff().dt.days
for s in SYS + ["any_defect"]:
    d["cur_" + s] = d[s]                                           # result AT inspection i
    d["cum_" + s] = g[s].cumsum()                                  # number of earlier defects of this system up to and including i
    d["last3_" + s] = d[s] + g[s].shift(1).fillna(0) + g[s].shift(2).fillna(0)
    d["next_" + s] = g[s].shift(-1)                                # TARGET: result at inspection i+1
d["next_date"] = g.date.shift(-1)
d["gap_next"] = (d.next_date - d.date).dt.days / 365.25            # planned interval to the next inspection (in Israel the registry gives the due date)
pairs = d[d.next_date.notna() & (d.gap_next >= 0.5)].copy()
pairs["target_date"] = pairs.next_date
print(f"inspections {len(d):,} of {d.unit.nunique():,} cars -> consecutive pairs {len(pairs):,} ({pairs.unit.nunique():,} cars)")
print("gap to next inspection (years):", pairs.gap_next.round().value_counts().sort_index().to_dict())
print("pairs by year of the TARGET inspection:", pairs.target_date.dt.year.value_counts().sort_index().to_dict())
print("prior inspections observed (n_insp) at the prediction time:", pairs.n_insp.value_counts().sort_index().to_dict())
top = pairs.make.value_counts().head(60).index; pairs["make_c"] = pairs.make.where(pairs.make.isin(top), "OTHER")
topm = pairs.model.value_counts().head(400).index; pairs["model_c"] = pairs.model.where(pairs.model.isin(topm), "OTHER")
pairs = pairs[pairs.fuel.isin(["petrol", "diesel", "hybrid", "electric"])]
CATS = {c: sorted(pairs[c].unique()) for c in ["fuel", "make_c", "model_c"]}
pairs["age_next"] = pairs.age_year + pairs.gap_next
pairs["dsp_f"] = pairs.days_since_prev.fillna(0); pairs["has_prev"] = pairs.days_since_prev.notna().astype(int)   # for Logistic Regression (no NaN support)
print("prevalence of the target at inspection i+1:", {s: round(float(pairs["next_" + s].mean()), 4) for s in SYS})
for s in SYS:
    print(f"  {s}: defect at i+1 given defect at i = {pairs.loc[pairs['cur_' + s] == 1, 'next_' + s].mean():.3f} | given no defect at i = {pairs.loc[pairs['cur_' + s] == 0, 'next_' + s].mean():.3f}")

inspections 812,021 of 371,250 cars -> consecutive pairs 434,027 (269,453 cars)
gap to next inspection (years): {1.0: 374195, 2.0: 59679, 3.0: 149, 4.0: 4}
pairs by year of the TARGET inspection: {2023: 10, 2024: 18291, 2025: 185968, 2026: 229758}
prior inspections observed (n_insp) at the prediction time: {1: 267035, 2: 158179, 3: 8707, 4: 104, 5: 2}


prevalence of the target at inspection i+1: {'brakes': 0.0959, 'electrical': 0.2114, 'engine_emissions': 0.0358}
  brakes: defect at i+1 given defect at i = 0.221 | given no defect at i = 0.084
  electrical: defect at i+1 given defect at i = 0.374 | given no defect at i = 0.171
  engine_emissions: defect at i+1 given defect at i = 0.157 | given no defect at i = 0.032


## 1. Do histories exist? Can vehicles be followed?
The first table shows that consecutive inspections of the same plate are available for most cars (the counts above). Two things are visible straight away: the target is **persistent** (a car with a defect in a system at inspection *i* is several times more likely to have one at *i+1*), and the open data only reach back to late 2023.

## 2. Variants (features) and protocol
| Variant | Features |
| --- | --- |
| **V0** registry at the last inspection | age at inspection *i*, fuel, make, model |
| **V0+due** | V0 plus the **planned interval to the next inspection** (age at the next inspection). In Israel the registry has the due date, so this is obtainable |
| **V1** | V0+due plus the result of the **same system** at inspection *i* |
| **V2** | V1 plus the full history summary: results of **all systems** at *i*, defects in the last three inspections, cumulative defects, number of inspections observed, days since the previous inspection |
Models: XGBoost on the natural objective; Logistic Regression (spline on age, one-hot make/fuel) as the baseline on V2. Metrics on the held-out future pairs of unseen cars; threshold for P / R / F1 chosen on the validation cars (never on test).

In [2]:
from sklearn.model_selection import GroupShuffleSplit
CUT = pd.Timestamp("2026-01-01")
VARS = {"V0": ["age_year", "fuel", "make_c", "model_c"],
        "V0+due": ["age_year", "gap_next", "fuel", "make_c", "model_c"],
        "V1": ["age_year", "gap_next", "fuel", "make_c", "model_c", "SELF"],
        "V2": ["age_year", "gap_next", "fuel", "make_c", "model_c", "SELF", "cur_brakes", "cur_electrical", "cur_engine_emissions", "cur_any_defect",
               "last3_SELF", "cum_SELF", "n_insp", "days_since_prev"]}
def cols_for(v, s):
    cs = [c.replace("SELF", s) if c in ("last3_SELF", "cum_SELF") else ("cur_" + s if c == "SELF" else c) for c in VARS[v]]
    return list(dict.fromkeys(cs))
def prep(df, cols):
    X = df[cols].copy()
    for c in cols:
        if c in CATS: X[c] = pd.Categorical(X[c], categories=CATS[c])
    return X
def fit_xgb(X, y):
    return XGBClassifier(n_estimators=250, max_depth=5, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, min_child_weight=50, tree_method="hist", enable_categorical=True, n_jobs=8, random_state=0).fit(X, y)
def capture(y, p, frac=0.2):
    k = int(len(p) * frac); idx = np.argsort(-p)[:k]; y = np.asarray(y); return float(y[idx].sum() / y.sum())
def ece(y, p, bins=10):
    q = pd.qcut(p, bins, duplicates="drop"); t = pd.DataFrame({"y": y, "p": p}).groupby(q, observed=True).agg(p=("p", "mean"), y=("y", "mean"), n=("y", "size"))
    return float((abs(t.p - t.y) * t.n).sum() / t.n.sum())
def best_f1_thr(y, p):
    pr, rc, th = precision_recall_curve(y, p); f1 = 2 * pr * rc / np.clip(pr + rc, 1e-12, None); return float(th[int(np.nanargmax(f1[:-1]))])
def lr_pipe(cols):
    num = [c for c in cols if c not in ("fuel", "make_c", "model_c", "age_year")]
    return Pipeline([("p", ColumnTransformer([("a", SplineTransformer(n_knots=6, degree=3), ["age_year"]), ("n", StandardScaler(), num),
                                              ("c", OneHotEncoder(handle_unknown="ignore", min_frequency=50, sparse_output=False), ["fuel", "make_c"])])), ("m", LogisticRegression(max_iter=2000))])

def run(seed):
    cars = pairs.unit.drop_duplicates().to_numpy(); rng = np.random.default_rng(seed); rng.shuffle(cars)
    n = len(cars); tr_c, va_c, te_c = set(cars[:int(.7 * n)]), set(cars[int(.7 * n):int(.85 * n)]), set(cars[int(.85 * n):])
    tr = pairs[pairs.unit.isin(tr_c) & (pairs.target_date < CUT)]; va = pairs[pairs.unit.isin(va_c) & (pairs.target_date < CUT)]
    te = pairs[pairs.unit.isin(te_c) & (pairs.target_date >= CUT)]; seen = pairs[pairs.unit.isin(tr_c) & (pairs.target_date >= CUT)]
    assert not (set(tr.unit) & set(te.unit)) and not (set(va.unit) & set(te.unit)) and tr.target_date.max() < te.target_date.min()
    out, extra = [], {}
    for s in SYS:
        t = "next_" + s
        for v in VARS:
            cols = cols_for(v, s); m = fit_xgb(prep(tr, cols), tr[t]); p = m.predict_proba(prep(te, cols))[:, 1]; pv = m.predict_proba(prep(va, cols))[:, 1]; y = te[t].to_numpy()
            thr = best_f1_thr(va[t], pv); pred = (p >= thr); tp = int((pred & (y == 1)).sum())
            out.append({"seed": seed, "system": s, "variant": v, "model": "XGBoost", "n test": len(te), "prevalence": y.mean(), "ROC-AUC": roc_auc_score(y, p), "PR-AUC": average_precision_score(y, p),
                        "capture 20%": capture(y, p), "ECE": ece(y, p), "threshold": thr, "flagged": pred.mean(), "Precision": tp / max(pred.sum(), 1), "Recall": tp / y.sum(), "F1": f1_score(y, pred)})
            if v == "V2":
                ps = m.predict_proba(prep(seen, cols))[:, 1]; extra[(s, "seen cars, 2026")] = roc_auc_score(seen[t], ps); extra[(s, "unseen cars, 2026")] = roc_auc_score(y, p)
                if seed == 0: extra[(s, "importance")] = pd.Series(m.feature_importances_, index=cols).sort_values(ascending=False).head(8).round(3).to_dict()
        cols = [c for c in cols_for("V2", s) if c != "days_since_prev"] + ["dsp_f", "has_prev"]; lr = lr_pipe(cols).fit(tr[cols].astype({"fuel": str, "make_c": str}), tr[t]); p = lr.predict_proba(te[cols].astype({"fuel": str, "make_c": str}))[:, 1]
        pv = lr.predict_proba(va[cols].astype({"fuel": str, "make_c": str}))[:, 1]; thr = best_f1_thr(va[t], pv); pred = (p >= thr); tp = int((pred & (y == 1)).sum())
        out.append({"seed": seed, "system": s, "variant": "V2", "model": "Logistic Regression", "n test": len(te), "prevalence": y.mean(), "ROC-AUC": roc_auc_score(y, p), "PR-AUC": average_precision_score(y, p),
                    "capture 20%": capture(y, p), "ECE": ece(y, p), "threshold": thr, "flagged": pred.mean(), "Precision": tp / max(pred.sum(), 1), "Recall": tp / y.sum(), "F1": f1_score(y, pred)})
        prev = y.mean(); out.append({"seed": seed, "system": s, "variant": "flag every car", "model": "baseline", "n test": len(te), "prevalence": prev, "ROC-AUC": 0.5, "PR-AUC": prev, "capture 20%": 0.2, "ECE": np.nan,
                                     "threshold": 0, "flagged": 1.0, "Precision": prev, "Recall": 1.0, "F1": 2 * prev / (1 + prev)})
    return out, extra
res, EX = [], {}
for seed in [0, 1, 2]:
    o, e = run(seed); res += o; EX[seed] = e
R = pd.DataFrame(res)
print(f"test pairs per split (unseen cars, target in 2026): {int(R['n test'].iloc[0]):,}")

test pairs per split (unseen cars, target in 2026): 34,418


## 3. Results (mean of 3 random car splits; ± = spread max-min)

In [3]:
agg = R.groupby(["system", "variant", "model"], sort=False).agg(**{"ROC-AUC": ("ROC-AUC", "mean"), "AUC spread": ("ROC-AUC", lambda x: x.max() - x.min()), "PR-AUC": ("PR-AUC", "mean"), "prevalence": ("prevalence", "mean"),
                                                                   "capture 20%": ("capture 20%", "mean"), "ECE": ("ECE", "mean"), "flagged": ("flagged", "mean"), "Precision": ("Precision", "mean"), "Recall": ("Recall", "mean"), "F1": ("F1", "mean")}).reset_index()
display(agg)

,system,variant,model,ROC-AUC,AUC spread,PR-AUC,prevalence,capture 20%,ECE,flagged,Precision,Recall,F1
0,brakes,V0,XGBoost,0.6693,0.0087,0.1589,0.0902,0.3867,0.0067,0.2750,0.1615,0.4923,0.2432
1,brakes,V0+due,XGBoost,0.6700,0.0075,0.1595,0.0902,0.3854,0.0071,0.2934,0.1591,0.5173,0.2432
2,brakes,V1,XGBoost,0.6830,0.0050,0.1814,0.0902,0.4064,0.0072,0.2085,0.1813,0.4191,0.2531
3,brakes,V2,XGBoost,0.6882,0.0046,0.1846,0.0902,0.4150,0.0083,0.2130,0.1842,0.4345,0.2583
4,brakes,V2,Logistic Regression,0.6865,0.0079,0.1921,0.0902,0.4151,0.0040,0.1996,0.1891,0.4166,0.2593
5,brakes,flag every car,baseline,0.5000,0.0000,0.0902,0.0902,0.2000,NaN,1.0000,0.0902,1.0000,0.1655
6,electrical,V0,XGBoost,0.6767,0.0034,0.3137,0.1976,0.3485,0.0078,0.3833,0.3101,0.6016,0.4092
7,electrical,V0+due,XGBoost,0.6773,0.0029,0.3142,0.1976,0.3486,0.0100,0.4279,0.3007,0.6502,0.4108
8,electrical,V1,XGBoost,0.6976,0.0010,0.3576,0.1976,0.3840,0.0093,0.3791,0.3188,0.6117,0.4191
9,electrical,V2,XGBoost,0.7050,0.0015,0.3675,0.1976,0.3982,0.0091,0.3669,0.3284,0.6089,0.4261


## 4. Leakage and memorisation check
AUC of V2 on the **future pairs of cars that WERE in training** (their earlier history was seen) against the unseen cars. A large difference would mean the model memorises cars.

In [4]:
rows = []
for s in SYS:
    rows.append({"system": s, "unseen cars, 2026": np.mean([EX[k][(s, "unseen cars, 2026")] for k in EX]), "training cars, 2026": np.mean([EX[k][(s, "seen cars, 2026")] for k in EX])})
display(pd.DataFrame(rows))
for s in SYS: print(s, "top features (gain share), seed 0:", EX[0][(s, "importance")])

,system,"unseen cars, 2026","training cars, 2026"
0,brakes,0.6882,0.6854
1,electrical,0.7050,0.7020
2,engine_emissions,0.7980,0.8008


brakes top features (gain share), seed 0: {'cur_brakes': 0.2879999876022339, 'last3_brakes': 0.20100000500679016, 'age_year': 0.15000000596046448, 'cur_any_defect': 0.12300000339746475, 'model_c': 0.06199999898672104, 'cum_brakes': 0.032999999821186066, 'make_c': 0.03200000151991844, 'cur_engine_emissions': 0.028999999165534973}
electrical top features (gain share), seed 0: {'cur_electrical': 0.5820000171661377, 'last3_electrical': 0.125, 'age_year': 0.08500000089406967, 'cur_any_defect': 0.07000000029802322, 'cum_electrical': 0.035999998450279236, 'model_c': 0.03099999949336052, 'make_c': 0.019999999552965164, 'cur_engine_emissions': 0.014999999664723873}
engine_emissions top features (gain share), seed 0: {'cur_engine_emissions': 0.2849999964237213, 'last3_engine_emissions': 0.23199999332427979, 'age_year': 0.15800000727176666, 'model_c': 0.06700000166893005, 'cur_any_defect': 0.06499999761581421, 'cum_engine_emissions': 0.04899999871850014, 'make_c': 0.03400000184774399, 'cur_electr

## 5. Gain from the history, with a paired view
Difference in mean AUC between variants (V0 is what the cross-sectional model of notebooks 04–05 would know about the car at the last inspection).

In [5]:
pv = R[R.model == "XGBoost"].pivot_table(index="system", columns="variant", values="ROC-AUC")[list(VARS)]
pv["V2 - V0"] = pv["V2"] - pv["V0"]; pv["V2 - V0+due"] = pv["V2"] - pv["V0+due"]
display(pv)

variant,V0,V0+due,V1,V2,V2 - V0,V2 - V0+due
system,,,,,,
brakes,0.6693,0.6700,0.6830,0.6882,0.0188,0.0181
electrical,0.6767,0.6773,0.6976,0.7050,0.0284,0.0278
engine_emissions,0.7842,0.7846,0.7933,0.7980,0.0138,0.0135
